# Short-Term Memory, Hands On

This notebook rebuilds the ideas in `demo.py` one at a time, so that each concept can be run and
watched rather than read. It is not a copy of `demo.py`: that file is a deployable agent, while this
is a lesson that happens to arrive at the same code.

**Part A — no AWS resources.** Steps 1 to 6 run entirely inside this kernel. They make real Bedrock
model calls, which cost a fraction of a cent, but they create nothing that has to be torn down.

**Part B — real AgentCore Memory.** Steps 7 onwards create a live memory resource. Not written yet.

Run the cells in order. Each step ends with output you can check against what the markdown said to
expect.

## Step 1 — Make the notebook runnable

`demo.py` opens with `BASE_DIR = Path(__file__).resolve().parent`. That line cannot work here:
**`__file__` exists only inside a file being executed as a script**, and a notebook cell is a string
handed to a kernel, so there is no file for it to name. Copying `demo.py`'s version into a cell
raises `NameError`.

The notebook equivalent is `Path.cwd()` — the directory the *kernel* was started in, which is
normally the one holding the notebook. Note that this is a weaker guarantee than `__file__`: it
depends on how the kernel was launched rather than on where the code lives. So instead of trusting
it, the cell below checks two candidate locations and prints which one it found, turning a silently
wrong path into a visible one.

> **Note:** `flights.json` was copied here from `lab_2`, and the `datasets` folder sits beside
> this notebook — which is also where `demo.py` looks for it, so both work from the same copy.
> The second candidate below is a safety net for a kernel started somewhere other than this
> directory.

Two further departures from `demo.py`, both deliberate:

- **No `app = BedrockAgentCoreApp()`.** That object exists to serve HTTP inside a deployed
  container. Nothing in Part A serves anything, so constructing it would leave an unused object on
  the page with no way to explain it.
- **Logging is configured for a kernel, not a container.** `basicConfig` is documented to do nothing
  when the root logger already has handlers, which a kernel may well have installed, so `force=True`
  makes the configuration apply regardless. The root stays at `WARNING` to keep botocore quiet and
  only our own logger goes to `INFO`.

**What to expect:** two version numbers, a resolved `datasets` path, and `flights.json` reporting 15
records. Anything else means the path resolution needs fixing before going any further.

In [1]:
# =====================================================
# 1. IMPORTS - only what Part A actually needs
# =====================================================
# strands.hooks and MemoryClient are introduced later, at the steps that explain
# them, rather than all at once here.
import json
import logging
import os
from importlib.metadata import version
from pathlib import Path

from strands import Agent, tool
from strands.models import BedrockModel

# =====================================================
# 2. CREDENTIAL RESOLUTION - must precede any boto3 client
# =====================================================
# botocore looks for its config and credentials files in a default location; these
# two variables redirect it to ~/.aws-personal/, where this project's credentials
# live. Neither line contains a credential - only the path to the file holding one.
#
# Placement is load-bearing rather than tidiness. BedrockModel.__init__ builds a
# boto3 Session and a client, and credentials are resolved at that moment - so
# this has to run before the model is created, one cell below. A SUCCESSFUL
# resolution is then cached on the session for its lifetime, so building the model
# first would bind it to whatever store happened to be in effect and reduce a
# later assignment here to a silent no-op. (A FAILED resolution is not cached:
# botocore stores None and cannot distinguish "found nothing" from "not looked
# yet". That is why setting these below the model appeared to work at first - the
# retry at request time picked them up. It only holds while the default location
# yields nothing, so it is not a property to rely on.)
#
# The alternative is exporting both in the shell that launches the IDE, which also
# fixes the `aws` and `agentcore` CLIs and every other notebook, and does not tie
# correctness to cell execution order. Done here instead to keep this notebook
# self-contained.
os.environ["AWS_CONFIG_FILE"] = os.path.expanduser("~/.aws-personal/config")
os.environ["AWS_SHARED_CREDENTIALS_FILE"] = os.path.expanduser("~/.aws-personal/credentials")

In [2]:
# =====================================================
# 3. LOGGING - for a kernel, not a container
# =====================================================
# force=True replaces whatever handler the kernel may already have installed;
# without it, basicConfig is documented to do nothing in that situation.
logging.basicConfig(
    level=logging.WARNING,               # keeps botocore and urllib3 quiet
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)
logger = logging.getLogger("WanderBot.Notebook")
logger.setLevel(logging.INFO)            # our own messages only

# =====================================================
# 4. PATHS - the notebook's replacement for __file__
# =====================================================
BASE_DIR = Path.cwd()

# Two candidates, checked in order: beside the notebook, then one level up in
# lab_4/. Whichever actually holds flights.json wins, and gets printed below, so
# a wrong guess shows up immediately instead of failing later inside a tool.
CANDIDATES = [BASE_DIR / "datasets", BASE_DIR.parent / "datasets"]
DATA_DIR = next((p for p in CANDIDATES if (p / "flights.json").is_file()), None)

# =====================================================
# 5. MODEL - one adapter, reused by every agent below
# =====================================================
# Built once: the adapter holds no conversation, so there is nothing per-agent
# about it. region_name and model_id are keyword-only on strands-agents 1.53.
REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

# =====================================================
# 6. PRINTING - keep saved output publishable
# =====================================================
# A notebook's outputs are stored inside the .ipynb and committed with it. So
# anything printed here is published, and two categories need care: absolute paths
# (which carry the machine's username and home layout) and AWS identifiers (account
# ids, ARNs, resource ids). Both go through a helper before being printed.


def repo_relative(path):
    """Render a path relative to the repository root.

    Walks up to the directory containing .git. This keeps the whole diagnostic
    value - which of the candidate directories actually won - while leaving the
    home prefix out of the saved output.
    """
    path = Path(path)
    for parent in [path, *path.parents]:
        if (parent / ".git").is_dir():
            return str(path.relative_to(parent))
    return path.name                      # not inside a repo: show the leaf only


def mask_id(value, keep_last=4):
    """Mask an identifier, keeping the final characters.

    Enough left to recognise which account or resource this is when comparing
    against the console, not enough to be reused by a reader of the repo.
    """
    s = str(value)
    if len(s) <= keep_last:
        return "*" * len(s)
    return "*" * (len(s) - keep_last) + s[-keep_last:]


def mask_arn(arn):
    """Mask an ARN's account id and unique suffix, keeping its structure.

    The service, region, resource type and human-chosen name are the parts worth
    reading while learning. The account and the generated suffix are the parts
    worth withholding, so only those are hidden.
    """
    parts = str(arn).split(":")
    if len(parts) < 6 or parts[0] != "arn":
        return mask_id(arn)
    parts[4] = mask_id(parts[4])                        # the account id
    kind, sep, name = parts[5].partition("/")           # e.g. runtime/WanderBot-AbC123
    if sep:
        head, dash, tail = name.rpartition("-")         # keep the name, mask the suffix
        parts[5] = f"{kind}/{head}{dash}{'*' * len(tail)}" if dash else f"{kind}/{mask_id(name)}"
    return ":".join(parts)


# =====================================================
# 7. VERIFY - every assumption the rest of the notebook rests on
# =====================================================
print("strands-agents   :", version("strands-agents"))
print("bedrock-agentcore:", version("bedrock-agentcore"))
print("kernel cwd       :", repo_relative(BASE_DIR))
print("datasets found at:", repo_relative(DATA_DIR))

if DATA_DIR is None:
    raise FileNotFoundError(
        # repo-relative here too: a traceback is saved output like any other
        f"No datasets/flights.json under any of: {[repo_relative(p) for p in CANDIDATES]}"
    )

flights = json.loads((DATA_DIR / "flights.json").read_text())
print("flight records   :", len(flights))
print("sample record    :", flights[0]["origin"], "->", flights[0]["destination"], flights[0]["date"])

# What a masked identifier looks like, so the convention is visible before Part B
# starts producing real ones. The account below is a placeholder, not an account.
EXAMPLE_ARN = "arn:aws:bedrock-agentcore:us-east-1:123456789012:runtime/WanderBot-AbCdEf1234"
print("masking example  :", mask_arn(EXAMPLE_ARN))

strands-agents   : 1.53.0
bedrock-agentcore: 1.22.0
kernel cwd       : lab_4/Demo
datasets found at: lab_4/Demo/datasets
flight records   : 15
sample record    : LHR -> CDG 2026-03-15
masking example  : arn:aws:bedrock-agentcore:us-east-1:********9012:runtime/WanderBot-**********


`demo.py`'s system prompt, verbatim. Nothing uses it until Step 2, where the first agent is built —
it lives here because it is configuration rather than a step in the argument.

In [3]:
SYSTEM_PROMPT = """You are WanderBot, the AI travel assistant for Horizon Travel.

You are helping a customer plan their trip across a multi-turn conversation.

CONVERSATION GUIDELINES
- Refer back to earlier parts of the conversation when relevant
- If the customer mentions a city, remember it for subsequent questions
- Be proactive: suggest related information
- Keep track of the customer's stated preferences
- When the customer asks follow-up questions, answer in context

TOOL USE
- Use search_flights when asked about flight options or availability
- Always use tools rather than guessing specific data

IATA CODE QUICK REFERENCE
LHR = London Heathrow | CDG = Paris Charles de Gaulle | JFK = New York JFK
MIA = Miami | LAX = Los Angeles | BCN = Barcelona | FCO = Rome Fiumicino"""


## Step 2 — Prove the model has no memory

**A language model has no memory. Every call is the first call.** What looks like continuity in a
chat interface is the framework *re-sending* the whole conversation each turn, so the model is not
remembering — it is being reminded, at full token cost, every time.

That is a claim, and it is testable. The experiment holds the questions fixed and changes only where
the conversation is kept:

| | Arrangement | Question 1 | Question 2 |
| --- | --- | --- | --- |
| **A** | two separately constructed `Agent` objects | flights LHR to CDG | which is cheapest? |
| **B** | one `Agent` object, called twice | flights LHR to CDG | which is cheapest? |

The tool earns its place here. *Which one is the cheapest?* refers to rows returned by
`search_flights`, so the agent cannot bluff from general knowledge — either it still has the flight
list or it does not.

**Predict before running.** Arrangement A's second answer cannot identify the flights: with no prior
messages the question has no referent, so expect a request for clarification or an invented route.
Arrangement B should name HZ-102 at $159.99, the cheaper of the two LHR to CDG flights on that date.

Note what is *not* changing between A and B — same model, same prompt, same tool, same questions,
same order. The only variable is whether the second question arrives with the first exchange
attached.

In [4]:
# =====================================================
# THE TOOL - demo.py's search_flights, with its debug logging trimmed
# =====================================================
# @tool reads the docstring and type hints and builds a JSON schema from them,
# which is shipped with every model request. The description is what the model
# reads when deciding whether to call this.
@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """
    Search for Horizon Travel flights by route and date.

    Args:
        origin      : IATA departure airport code (e.g. 'LHR', 'JFK')
        destination : IATA arrival airport code (e.g. 'CDG', 'FCO')
        date        : Date in YYYY-MM-DD format
    Returns:
        Formatted list of matching flights with prices and availability.
    """
    logger.info("search_flights(origin=%s, destination=%s, date=%s)", origin, destination, date)

    records = json.loads((DATA_DIR / "flights.json").read_text())
    matches = [
        fl for fl in records
        if fl["origin"].upper() == origin.upper()
        and fl["destination"].upper() == destination.upper()
        and fl["date"] == date
    ]
    if not matches:
        return f"No flights found from {origin} to {destination} on {date}."

    out = [f"Flights {origin.upper()} -> {destination.upper()} on {date}:"]
    for fl in matches:
        out.append(
            f"  {fl['flight_number']} | {fl['departure_time']}->{fl['arrival_time']} "
            f"| {fl['cabin_class']} | ${fl['price_usd']} | {fl['available_seats']} seats "
            f"| {fl['status']}"
        )
    return "\n".join(out)


# tool_name comes from the decorator, which wraps the function in a
# DecoratedFunctionTool rather than leaving it a plain function.
print("tool ready:", search_flights.tool_name)

tool ready: search_flights


In [5]:
# =====================================================
# ARRANGEMENT A - two separately constructed agents
# =====================================================
# Every Agent() call produces a fresh object with an empty message list. This is
# not a contrived setup: a deployed AgentCore agent builds its agent inside the
# request handler, so it does exactly this on every invocation.

Q1 = "Are there any flights from LHR to CDG on 2026-03-15?"
Q2 = "Which one is the cheapest?"


def build_agent():
    """A brand-new agent: same model, same prompt, same tool, no history."""
    return Agent(model=model, system_prompt=SYSTEM_PROMPT, tools=[search_flights])


agent_one = build_agent()
print(f"Q1 -> {Q1}\n")
_ = agent_one(Q1)

print("\n" + "=" * 78)

agent_two = build_agent()          # a different object; nothing carried across
print(f"\nQ2 asked of a BRAND-NEW agent -> {Q2}\n")
_ = agent_two(Q2)

print("\n\nmessages held by agent_one:", len(agent_one.messages))
print("messages held by agent_two:", len(agent_two.messages))

Q1 -> Are there any flights from LHR to CDG on 2026-03-15?



2026-09-25 16:07:36,132 [INFO] WanderBot.Notebook: search_flights(origin=LHR, destination=CDG, date=2026-03-15)



Tool #1: search_flights
I found two flights from London Heathrow (LHR) to Paris Charles de Gaulle (CDG) on March 15, 2026:

1. **Flight HZ-101**
   - Departure: 06:30 AM
   - Arrival: 09:00 AM
   - Class: Economy
   - Price: $189.99
   - Seats available: 42
   - Status: Scheduled

2. **Flight HZ-102**
   - Departure: 12:45 PM
   - Arrival: 03:15 PM
   - Class: Economy
   - Price: $159.99
   - Seats available: 18
   - Status: Scheduled

Would you like help booking one of these flights or do you need information about return flights?

Q2 asked of a BRAND-NEW agent -> Which one is the cheapest?

I can help you find the cheapest flight option, but I need a bit more information first. Could you please tell me:

1. Your departure city (origin)
2. Your destination city
3. Your travel date

Once you provide these details, I'll search for available flights and show you the pricing options so we can identify the cheapest one for you.

messages held by agent_one: 4
messages held by agent_two: 2


In [6]:
# =====================================================
# ARRANGEMENT B - one agent, called twice
# =====================================================
agent_b = build_agent()

print(f"Q1 -> {Q1}\n")
result_1 = agent_b(Q1)
# Snapshot the usage now: accumulated_usage keeps adding up over the agent's
# life, so reading it later would show the total rather than this call's figure.
usage_after_q1 = dict(result_1.metrics.accumulated_usage)
print("\nmessages after Q1:", len(agent_b.messages))

print("\n" + "=" * 78)

print(f"\nQ2 asked of the SAME agent -> {Q2}\n")
result_2 = agent_b(Q2)
usage_after_q2 = dict(result_2.metrics.accumulated_usage)
print("\nmessages after Q2:", len(agent_b.messages))

Q1 -> Are there any flights from LHR to CDG on 2026-03-15?



2026-09-25 16:07:38,824 [INFO] WanderBot.Notebook: search_flights(origin=LHR, destination=CDG, date=2026-03-15)



Tool #1: search_flights
I found two flights from London Heathrow (LHR) to Paris Charles de Gaulle (CDG) on March 15, 2026:

1. **HZ-101** - Departs 06:30, arrives 09:00  
   Economy: $189.99 | 42 seats available | SCHEDULED

2. **HZ-102** - Departs 12:45, arrives 15:15  
   Economy: $159.99 | 18 seats available | SCHEDULED

Would you like help booking one of these flights or do you need any additional information about them?
messages after Q1: 4


Q2 asked of the SAME agent -> Which one is the cheapest?

The cheapest flight from LHR to CDG on March 15, 2026 is **HZ-102**, which departs at 12:45 and arrives at 15:15. It's available for **$159.99** in economy class, with 18 seats still available.

Would you like to proceed with booking this flight or would you like to know more details?
messages after Q2: 6


In [7]:
# =====================================================
# WHAT WAS ACTUALLY RE-SENT
# =====================================================
# In arrangement B the memory is agent.messages - a plain Python list living on
# the object. Nothing persistent and nothing clever: when the object goes, it
# goes. Note the content BLOCK types as well as the roles; a turn that uses a
# tool is several messages, not one.


def show_messages(agent):
    for i, m in enumerate(agent.messages):
        blocks = [next(iter(b)) for b in m.get("content", [])]
        preview = ""
        for b in m.get("content", []):
            if "text" in b:
                preview = " ".join(b["text"].split())[:64]
                break
        print(f"  [{i}] {m['role']:<9} {str(blocks):<26} {preview}")


print("agent_b.messages - the agent that answered:")
show_messages(agent_b)

print("\nagent_two.messages - the agent that could not:")
show_messages(agent_two)

# The cost of being reminded. accumulated_usage is what the model provider
# reported, so the delta is the real bill for re-sending the first exchange.
print("\nusage after Q1:", usage_after_q1)
print("usage after Q2:", usage_after_q2)
delta_in = usage_after_q2["inputTokens"] - usage_after_q1["inputTokens"]
print(f"\ninput tokens billed for Q2 alone: {delta_in}")
print(f"words actually typed in Q2      : {len(Q2.split())}")

# Do NOT read Q1 vs Q2 call-for-call. The message dump above shows TWO assistant
# messages for Q1 - a toolUse and then the text - because a tool-using turn needs
# a second model call to interpret the tool result. Q1's figure therefore covers
# two calls and Q2's covers one. The number that matters is delta_in set against
# those few typed words.

agent_b.messages - the agent that answered:
  [0] user      ['text']                   Are there any flights from LHR to CDG on 2026-03-15?
  [1] assistant ['toolUse']                
  [2] user      ['toolResult']             
  [3] assistant ['text']                   I found two flights from London Heathrow (LHR) to Paris Charles 
  [4] user      ['text']                   Which one is the cheapest?
  [5] assistant ['text']                   The cheapest flight from LHR to CDG on March 15, 2026 is **HZ-10

agent_two.messages - the agent that could not:
  [0] user      ['text']                   Which one is the cheapest?
  [1] assistant ['text']                   I can help you find the cheapest flight option, but I need a bit

usage after Q1: {'inputTokens': 2467, 'outputTokens': 180, 'totalTokens': 2647}
usage after Q2: {'inputTokens': 3912, 'outputTokens': 257, 'totalTokens': 4169}

input tokens billed for Q2 alone: 1445
words actually typed in Q2      : 5


### What the output proved

**Same model, same prompt, same tool, same question — different answer.** The only variable was
whether the second question arrived with the first exchange attached. Arrangement A asked for
departure city, destination and date; arrangement B named HZ-102 at $159.99. So continuity is not a
property of the model, and losing it is not a bug in the agent.

**The memory in arrangement B is `agent.messages`, an ordinary Python list on the object.** Nothing
persistent, nothing clever — when the object is garbage collected, the conversation is gone. That is
precisely why a deployed agent forgets: its handler constructs a new `Agent` per request, exactly as
`build_agent()` did twice here.

**A two-question conversation is six messages, not four.** The dump shows why: a turn that uses a
tool expands into four messages rather than one.

| Index | Role | Content block | What it is |
| --- | --- | --- | --- |
| 0 | `user` | `text` | the question |
| 1 | `assistant` | `toolUse` | the model asking for `search_flights` |
| 2 | `user` | `toolResult` | the flight rows going back in |
| 3 | `assistant` | `text` | the answer built from them |
| 4 | `user` | `text` | the follow-up |
| 5 | `assistant` | `text` | the answer, resolvable only because 0-3 are present |

Note row 2: **tool output is handed back to the model with the role `user`**, not some dedicated tool
role. The framework speaks as the user when returning results.

**Being reminded has a bill.** Q2 was five typed words and was charged about 1,500 input tokens, because the
whole prior exchange, the system prompt and the tool's JSON schema were all re-sent with it. Nothing
was cached and nothing was summarised — the model read the lot again from scratch.

That last point is what forces the next step. If history is a list that grows without bound, and
every turn re-sends all of it, then something has to decide what to drop. That decision is the
conversation manager, and one is already running whether or not you asked for it.

## Step 3 — Something is already throwing your history away

Step 2 ended on a problem: history is a list that grows every turn, and every turn re-sends all of it.
That cannot continue indefinitely — eventually the prompt exceeds the model's context window. So
something has to decide what to drop, and **that something is already running in every agent you have
built so far, whether or not you asked for it.**

Strands offers three strategies for the trade between context fidelity and tokens:

| Strategy | What it does | Class | Cost |
| --- | --- | --- | --- |
| **Full history** | send everything, every turn | no manager — you opt out of reduction | unbounded; eventually overflows |
| **Sliding window** | keep the most recent N messages | `SlidingWindowConversationManager` — **the default** | bounded, but old context is gone for good |
| **Summarisation** | condense older messages into a précis | `SummarizingConversationManager` | bounded and keeps the gist, at the price of an extra model call and lost detail |

Two details from the SDK source are worth getting right, because both are easy to misremember.

**`window_size` counts messages, not turns and not exchanges.** The docstring is explicit — *maximum
number of messages to keep* — and `apply_management` gates on `len(messages) <= window_size`. The
default is 40. Step 2 showed that a turn using a tool expands into four messages, so the default is
nearer ten tool-using turns than forty of anything.

**Reduction is in-place and permanent.** `reduce_context` assigns to `messages[:]`, mutating the list
the agent holds. There is no archive and no undo: evicted context is simply gone, which is exactly why
AgentCore Memory later becomes necessary rather than merely convenient.

The next three cells cost nothing, because `apply_management` can be called directly on a
hand-populated list rather than paying a model to generate one.

In [8]:
# =====================================================
# WHAT YOU GET WITHOUT ASKING
# =====================================================
# Every Agent built so far was given a conversation manager implicitly. Building
# one costs nothing: constructing an Agent makes no API call.
plain = Agent(model=model, system_prompt="unused")

cm = plain.conversation_manager
print("manager class :", type(cm).__name__)
print("window_size   :", cm.window_size, "messages")
print("pin_first     :", cm.pin_first)
print("per_turn      :", cm.per_turn)

# per_turn=False means management runs at the END of an invocation, not before
# each model call - so a single multi-tool turn can transiently hold more than
# window_size messages.
print("\nSo every agent in Step 2 was already discarding history once it passed",
      cm.window_size, "messages.")

manager class : SlidingWindowConversationManager
window_size   : 40 messages
pin_first     : None
per_turn      : False

So every agent in Step 2 was already discarding history once it passed 40 messages.


In [9]:
# =====================================================
# EVICTION, WITH NO MODEL INVOLVED
# =====================================================
# apply_management() takes an agent and rewrites agent.messages in place. Nothing
# stops us handing it a conversation we wrote ourselves, which makes the behaviour
# deterministic and free to watch.
from strands.agent.conversation_manager import SlidingWindowConversationManager


def msg(role, text):
    """A message in the shape Strands uses: a role plus a list of content blocks."""
    return {"role": role, "content": [{"text": text}]}


# A six-message window, then a ten-message conversation to overflow it.
narrow = Agent(
    model=model,
    system_prompt="unused",
    conversation_manager=SlidingWindowConversationManager(window_size=6),
)

narrow.messages[:] = [
    msg("user", "1. I want to plan a trip to Rome"),
    msg("assistant", "2. Lovely - when are you travelling?"),
    msg("user", "3. Mid March, flying from London"),
    msg("assistant", "4. Noted: LHR to Rome, March"),
    msg("user", "5. Two adults, one child"),
    msg("assistant", "6. Noted: party of three"),
    msg("user", "7. Budget is about 2000 pounds"),
    msg("assistant", "8. Noted: 2000 GBP"),
    msg("user", "9. We would prefer a morning flight"),
    msg("assistant", "10. Noted: mornings preferred"),
]

print("before :", len(narrow.messages), "messages")
show_messages(narrow)

narrow.conversation_manager.apply_management(narrow)

print("\nafter  :", len(narrow.messages), "messages")
show_messages(narrow)
print("\nremoved by the manager:", narrow.conversation_manager.removed_message_count)

before : 10 messages
  [0] user      ['text']                   1. I want to plan a trip to Rome
  [1] assistant ['text']                   2. Lovely - when are you travelling?
  [2] user      ['text']                   3. Mid March, flying from London
  [3] assistant ['text']                   4. Noted: LHR to Rome, March
  [4] user      ['text']                   5. Two adults, one child
  [5] assistant ['text']                   6. Noted: party of three
  [6] user      ['text']                   7. Budget is about 2000 pounds
  [7] assistant ['text']                   8. Noted: 2000 GBP
  [8] user      ['text']                   9. We would prefer a morning flight
  [9] assistant ['text']                   10. Noted: mornings preferred

after  : 6 messages
  [0] user      ['text']                   5. Two adults, one child
  [1] assistant ['text']                   6. Noted: party of three
  [2] user      ['text']                   7. Budget is about 2000 pounds
  [3] assistant ['te

In [10]:
# =====================================================
# pin_first - protecting the turn that set up the conversation
# =====================================================
# A plain sliding window will happily evict message 1, which is usually the one
# establishing what the whole conversation is about. pin_first=N protects the
# first N messages permanently; the pin is written on the first reduction and
# stays put.
pinned = Agent(
    model=model,
    system_prompt="unused",
    conversation_manager=SlidingWindowConversationManager(window_size=6, pin_first=2),
)

pinned.messages[:] = [
    msg("user", "1. I want to plan a trip to Rome"),        # <- the premise
    msg("assistant", "2. Lovely - when are you travelling?"),
    msg("user", "3. Mid March, flying from London"),
    msg("assistant", "4. Noted: LHR to Rome, March"),
    msg("user", "5. Two adults, one child"),
    msg("assistant", "6. Noted: party of three"),
    msg("user", "7. Budget is about 2000 pounds"),
    msg("assistant", "8. Noted: 2000 GBP"),
    msg("user", "9. We would prefer a morning flight"),
    msg("assistant", "10. Noted: mornings preferred"),
]

pinned.conversation_manager.apply_management(pinned)

print("after reduction with pin_first=2:", len(pinned.messages), "messages")
show_messages(pinned)
print("\nCompare with the run above: 'Rome' survives here and did not there.")

after reduction with pin_first=2: 8 messages
  [0] user      ['text']                   1. I want to plan a trip to Rome
  [1] assistant ['text']                   2. Lovely - when are you travelling?
  [2] user      ['text']                   5. Two adults, one child
  [3] assistant ['text']                   6. Noted: party of three
  [4] user      ['text']                   7. Budget is about 2000 pounds
  [5] assistant ['text']                   8. Noted: 2000 GBP
  [6] user      ['text']                   9. We would prefer a morning flight
  [7] assistant ['text']                   10. Noted: mornings preferred

Compare with the run above: 'Rome' survives here and did not there.


In [11]:
# =====================================================
# DOES IT ACTUALLY FIRE IN A REAL LOOP?
# =====================================================
# The cells above called apply_management by hand. This one never mentions it: a
# window of 4, three ordinary turns, and the count should plateau on its own.
#
# No tools on this agent deliberately - a tool-using turn is four messages, which
# would obscure the arithmetic. Without tools each turn is exactly two.
real = Agent(
    model=model,
    system_prompt="You are a concise travel assistant. Answer in one short sentence.",
    conversation_manager=SlidingWindowConversationManager(window_size=4),
)

for turn, question in enumerate(
    ["I am planning a trip to Rome.",
     "What is the weather like there in March?",
     "How many days should I stay?"],
    start=1,
):
    real(question)
    print(f"\n-- after turn {turn}: {len(real.messages)} messages in history")

print("\nfinal history:")
show_messages(real)
print("\nremoved by the manager:", real.conversation_manager.removed_message_count)

Enjoy your trip to Rome—don’t miss the Colosseum, Vatican City, and a delicious gelato!
-- after turn 1: 2 messages in history
In March, Rome has mild temperatures, ranging from 12°C to 18°C, with occasional rain.
-- after turn 2: 4 messages in history
A comfortable trip to Rome can be enjoyed in about 5 to 7 days.
-- after turn 3: 4 messages in history

final history:
  [0] user      ['text']                   What is the weather like there in March?
  [1] assistant ['text']                   In March, Rome has mild temperatures, ranging from 12°C to 18°C,
  [2] user      ['text']                   How many days should I stay?
  [3] assistant ['text']                   A comfortable trip to Rome can be enjoyed in about 5 to 7 days.

removed by the manager: 2


### What the output proved

**The manager was never optional.** A plain `Agent` reports a
`SlidingWindowConversationManager` with `window_size=40`, so every agent in Step 2 was already
discarding history — it simply never got long enough to notice.

**Eviction is silent, lossy and permanent.** The narrow-window run dropped the opening messages with no
warning, no return value to check and nothing left behind. An agent that has been trimmed does not know
it: it cannot tell the difference between a fact never mentioned and one evicted twenty turns ago.

**`pin_first` exists because the first turn is usually the most valuable.** It is also the most
vulnerable, being oldest. Pinning it is what stops a sliding window discarding the premise of the
conversation while dutifully preserving small talk from two turns ago. Note that the pin is applied on
the first reduction and then stays — it is not re-evaluated as the conversation grows.

**Pinned messages sit outside the window budget, not inside it.** `window_size=6` with `pin_first=2`
left **eight** messages, not six — the two pinned plus a full six-message window. So `pin_first` raises
the ceiling rather than spending part of it, which matters when sizing a window against a context limit.

**Reduction may also miss `window_size` for a second reason.** `reduce_context` explicitly avoids
leaving a `toolResult` without its `toolUse`, or the reverse, since either is an invalid conversation
for the model. It trims to a valid boundary rather than a precise count.

**`per_turn=False` is visible in the real run, not just a setting.** Turn 3 answered *"at least 5 days
to fully explore Rome's top attractions"* — correctly naming Rome — and yet the final history no longer
contains the message that mentioned Rome. Both are true because management runs at the *end* of an
invocation: the premise was present while the model was called and evicted immediately afterwards. A
fourth turn would have found no Rome at all. That gap between what the model saw and what survives is
the subtlest thing in this step.

Now put Steps 2 and 3 together, because they are two different failures that look identical from the
outside:

| | What was lost | Why |
| --- | --- | --- |
| **Step 2** | everything | the `Agent` object was new, so the list started empty |
| **Step 3** | the oldest part | the object survived, but the manager trimmed the list |

Both end with a customer repeating themselves. Neither is fixable by making the window larger, since
that only postpones it and costs more tokens per turn in the meantime. What is needed is history kept
*somewhere other than the agent object* — durable across invocations, and not subject to eviction
because the window filled. That is AgentCore Memory — but before any of it can be wired up, the code
doing the remembering has to know *whose* conversation it is looking at, and that is Step 4.

## Step 4 — Where does a session id live?

Step 3 left us needing history kept somewhere other than the agent object. Before that can be wired up,
whatever does the keeping has to answer two questions on every single invocation: **which user is this,
and which conversation is this?** Those two identifiers have to live somewhere.

The obvious place is the wrong one. Put them in the conversation and the model reads them as part of the
prompt — paying tokens every turn for a UUID it has no use for — and worse, they become evictable. The
sliding window from Step 3 would eventually discard the very identifiers the memory lookup depends on.

Strands separates three kinds of state. The course splits this two ways, transient versus session, and
the SDK's finer split is the more useful one:

| | Holds | Lifetime | Does the model see it? |
| --- | --- | --- | --- |
| **Conversation history** | the messages, in `agent.messages` | the conversation, minus whatever the manager evicts | **yes** — it *is* the prompt |
| **Agent state** | arbitrary JSON-serialisable data, in `agent.state` | the agent object's life, and beyond it if a session manager is attached | no |
| **Invocation state** | the caller's keyword arguments for one call | that one invocation | no |

**The axis that matters is visibility, not lifetime.** Conversation history is memory precisely *because*
it is re-sent. Agent state is useful precisely because it is *not*: a user id, a feature flag or a
running counter is data your code needs and the model has no business reading, and keeping it out of the
prompt costs nothing per turn.

`Agent(state=...)` accepts a plain dict or an `AgentState`, and the whole API is three methods —
`get(key)`, `set(key, value)`, `delete(key)`. Calling `get()` with no argument returns the entire dict.

Invocation state is the thinnest of the three and gets no demo here: it is the keyword arguments handed
to `agent(...)` for a single call, and a tool reaches it by asking for a context parameter —
`@tool(context=True)` hands the tool a `tool_context` carrying `tool_use`, `agent` and
`invocation_state`. That suits request-scoped things such as a trace id. It is the wrong home for a
session id, which would then have to be re-supplied on every call.

In [12]:
# =====================================================
# AGENT STATE - where the memory hook will find its two ids
# =====================================================
# This is exactly the shape demo.py uses: the entrypoint puts both ids into state,
# and the hook provider reads them back out. Note that nothing here touches the
# conversation.
stateful = Agent(
    model=model,
    system_prompt="You are a concise assistant. Answer in one short sentence.",
    state={"actor_id": "alice", "session_id": "session-2026-03-15-001"},
)

print("actor_id    :", stateful.state.get("actor_id"))
print("session_id  :", stateful.state.get("session_id"))
print("whole state :", stateful.state.get())          # get() with no key = everything

# State is not a message, so the conversation is still empty.
print("\nmessages in history:", len(stateful.messages))

# And it is writable mid-run, which is what makes it useful for counters and flags.
stateful.state.set("turn_counter", 0)
stateful.state.set("turn_counter", stateful.state.get("turn_counter") + 1)
print("turn_counter:", stateful.state.get("turn_counter"))
print("state keys  :", sorted(stateful.state.get().keys()))

actor_id    : alice
session_id  : session-2026-03-15-001
whole state : {'actor_id': 'alice', 'session_id': 'session-2026-03-15-001'}

messages in history: 0
turn_counter: 1
state keys  : ['actor_id', 'session_id', 'turn_counter']


In [13]:
# =====================================================
# IS IT REALLY INVISIBLE TO THE MODEL?
# =====================================================
# The table says the model cannot see agent state. That is worth proving rather
# than trusting: put a value in state that the model has never been told, then ask
# for it directly.
stateful.state.set("booking_reference", "HZ-REF-88213")

_ = stateful("What is my booking reference? Give the reference, or say you do not know it.")

print("\n--- everything the model was actually sent ---")
show_messages(stateful)

print("\nstill sitting in state, unread:", stateful.state.get("booking_reference"))
print("state keys the model never saw :", sorted(stateful.state.get().keys()))

I don't have access to personal data, including booking references, so I cannot provide your booking reference. 

For your booking details, please check your confirmation email or contact the service provider directly.
--- everything the model was actually sent ---
  [0] user      ['text']                   What is my booking reference? Give the reference, or say you do 
  [1] assistant ['text']                   I don't have access to personal data, including booking referenc

still sitting in state, unread: HZ-REF-88213
state keys the model never saw : ['actor_id', 'booking_reference', 'session_id', 'turn_counter']


### What the output proved

**Agent state really is invisible to the model.** Asked directly for its booking reference, the model
answered that it has no access to that data — and the message dump shows why rather than leaving it to
trust: two messages, the question and the reply, neither containing `HZ-REF-88213`. At that moment state
held four keys and not one of them reached the prompt.

That is the property which makes state the right home for the two memory identifiers. They cost nothing
per turn, and they cannot be evicted by the sliding window from Step 3, because they were never in the
conversation to begin with.

**But state does not solve persistence, and it is important not to over-read it.** `agent.state` lives on
the agent object. Construct a new `Agent` and you get a new, empty `AgentState` — the same fate as the
message list in Step 2. So state is where the ids live *during* an invocation; it is not a store that
survives one. Which is why `demo.py`'s entrypoint sets them on every single invocation:

``` Python
agent = Agent(
    ...,
    state={"session_id": session_id, "actor_id": actor_id},
)
```

`session_id` comes from `context`, supplied by AgentCore Runtime and shared by every invocation in the
same session. `actor_id` comes from the payload, so the caller passes it. Both go into state, and the
hook provider reads them back out from there.

So the pieces are now in place and the gap is narrow. There is history that needs to outlive the agent
object, and there are two identifiers naming what to store it under. What is missing is the *timing*:
something has to run at the start of an invocation to read prior turns back, and again on every message
to write new ones out. Neither moment is reachable from the entrypoint, which only gets control before
and after the whole call.

Those moments are what the Strands hook system exposes, and that is Step 5.

## Step 5 — Running your own code inside the agent loop

**A hook is a callback the Agent invokes at a named point in its own lifecycle.** Calling an `Agent` runs a closed loop: append the user message, call the model, run any tools it asked for, append the results, repeat until the model stops. Hooks are the SDK's supported way to inject behaviour into that loop *without* subclassing `Agent` or editing the SDK. The mechanism is publish/subscribe with types — the agent emits **event objects**, and you register **callbacks** against the event **classes** you care about.

In `strands-agents` 1.53.0 the module exports 15 concrete event types: invocation start/end, before/after model call, before/after tool call, message added, agent initialised, plus multi-agent equivalents. `demo.py` subscribes to exactly two of them; the cells below watch eight, so that the ordering becomes visible rather than assumed.

| Import | What it is | Why this demo needs it |
| --- | --- | --- |
| `HookProvider` | A `@runtime_checkable` **Protocol** with a single method: `register_hooks(self, registry, **kwargs) -> None` | The packaging unit. One class bundles the two callbacks *and* the state they share — the `MemoryClient`, the `memory_id`, and `last_k_turns`. `Agent(hooks=[...])` accepts `HookProvider` instances (or bare callbacks) and for each provider calls `registry.add_hook(provider)`, which does nothing but call back into `provider.register_hooks(registry)`. Because it is a Protocol, inheriting from it is optional — any object with that method works — but subclassing documents the intent and gets type checking |
| `HookRegistry` | The switchboard the Agent owns as `agent.hooks`: a list of callbacks per event type, sorted by an `order` number, lower first | Never constructed by us. It arrives as the argument to `register_hooks`, where `registry.add_callback(EventType, method)` is the only call made on it; in the signature it serves purely as the type annotation. The agent registers its own built-ins on this same registry — conversation manager, retry strategy, session manager — so our callbacks queue alongside them |
| `AgentInitializedEvent` | Fired once, on the final line of `Agent.__init__`, **synchronously**. Its only field is `agent` | The **read** side of memory. This is the one moment when the agent exists but has not yet called the model, so it is the last chance to alter `system_prompt` before the first inference. The hook uses it to pull the last *k* turns out of AgentCore Memory and append them to the prompt |
| `MessageAddedEvent` | Fired inside `Agent._append_messages`, once per message appended, **asynchronously**. Fields: `agent` and `message` | The **write** side. Every user message, assistant reply and tool result is persisted to AgentCore Memory as it is added, so the next invocation has something to read back |

**The two events are a pair, and that pairing is the whole trick.** A deployed AgentCore agent constructs a fresh `Agent` object on every invocation, so nothing survives in process memory. `MessageAddedEvent` writes each turn out to a durable store during invocation *n*; `AgentInitializedEvent` reads them back at the start of invocation *n+1* and re-injects them as text. The agent is still stateless — the appearance of memory comes entirely from re-hydrating the prompt.

### Two rules that look arbitrary until you see the reason

**Rule 1 — the initialisation callback must be a plain `def`.** Register an `async def` for
`AgentInitializedEvent` and `add_callback` refuses it outright, with
`ValueError: AgentInitializedEvent can only be registered with a synchronous callback`.

The reason is that this event is fired from inside `Agent.__init__`, using the *synchronous*
`invoke_callbacks()`. A Python constructor cannot `await`, so there is no opportunity to run a coroutine
there. Had the SDK allowed it, calling your `async def` would merely have produced a coroutine object that
nobody awaits: the body would never execute, and the only clue would be a `coroutine was never awaited`
warning somewhere in stderr. Refusing it at registration converts an invisible bug into an immediate one.

The consequence you actually feel is this. Reading prior turns back from AgentCore Memory is a network
call, and it has to happen inside a plain `def` — even though `demo.py`'s entrypoint is `async def`. So
`get_last_k_turns` blocks the event loop while it runs. Tolerable for one call at the start of an
invocation; not something to put in a loop.

**Rule 2 — events are read-only *by default*, and each event opts individual fields back in.**
`BaseHookEvent.__setattr__` rejects every write unless the event class names that field as writable
through `_can_write`. So read-only is the floor rather than the whole story, and the fields an event does
open are the SDK's deliberate control surface — the levers a hook is meant to pull:

| Event | Fields you may write |
| --- | --- |
| `BeforeInvocationEvent` | `messages`, `cancel` |
| `BeforeModelCallEvent` | `cancel` |
| `AfterModelCallEvent` | `retry` |
| `BeforeToolCallEvent` | `cancel_tool`, `selected_tool`, `tool_use` |
| `AfterToolCallEvent` | `result`, `retry` |
| `BeforeToolsEvent` | `cancel` |
| `AfterToolsEvent` | `end_turn` |
| `AfterInvocationEvent` | `resume` |
| **`AgentInitializedEvent`** | **nothing** |
| **`MessageAddedEvent`** | **nothing** |

The last two rows are the ones that matter here. **Both events used for memory open nothing at all** —
they are pure notifications, with no field a hook can influence. Which is why the read hook does not
modify its event; it *cannot*. It reaches through it instead:

``` Python
event.message = new_message              # AttributeError: Property message is not writable
event.agent.system_prompt += "..."       # allowed
```

Both lines begin with `event`, which is what makes the difference easy to miss. The first **writes to the
event**. The second only **reads** `event.agent` — reading is never restricted — and then writes to the
agent object it hands back. The event itself is untouched afterwards: it still points at the same agent.
The freeze protects the notification, not the thing being notified about.

And the reason for freezing at all: a single event object is broadcast to every registered callback in
turn. If the first callback could rewrite it, later callbacks would see something different from what
actually fired, and behaviour would start depending on registration order. Freezing the event guarantees
every subscriber sees the same thing.

### 5a. Look inside the registry before adding anything

`agent.hooks` is a `HookRegistry`, and internally it keeps one list of callbacks per event type. The cell
below prints that structure for an agent built with **no** `hooks` argument at all.

What to look for: the registry is not empty. Strands wires its own components in through the very same
mechanism you are about to use, so you will see `SlidingWindowConversationManager` from Step 3 among them.
Note the `order` numbers too — one component deliberately runs last.

No model call, so this cell is free.

In [14]:
# =====================================================
# 5a. WHAT IS IN THE REGISTRY ALREADY
# =====================================================


def describe_owner(callback):
    """Say which object a registered callback belongs to.

    Args:
        callback: A function or bound method taken from the hook registry.

    Returns:
        str: The owning class's name when the callback is a bound method, for
            example "SlidingWindowConversationManager". The string
            "plain function" when the callback belongs to no object.
    """
    # Python attaches the instance to a bound method as __self__. A plain
    # function has no such attribute.
    owner = getattr(callback, "__self__", None)

    if owner is None:
        return "plain function"
    return type(owner).__name__


# An ordinary agent. Note there is no hooks= argument anywhere.
bare = Agent(model=model, system_prompt="unused")

# _registered_callbacks is private - there is no public API for reading the
# registry - so we are peeking deliberately to see how Strands assembles itself.
registered = bare.hooks._registered_callbacks

count = 0
for event_type in registered:
    print(event_type.__name__)

    for entry in registered[event_type]:
        owner = describe_owner(entry.callback)
        print("    order=" + str(entry.order), " ", owner + "." + entry.callback.__name__)
        count += 1

print()
print(count, "callbacks were already registered, and none of them are ours.")
print("SlidingWindowConversationManager is the conversation manager from Step 3:")
print("it is a HookProvider too, and it compresses history from BeforeModelCallEvent.")

BeforeModelCallEvent
    order=0   SlidingWindowConversationManager._on_before_model_call_threshold
    order=0   SlidingWindowConversationManager._on_before_model_call
    order=0   AgentDelegation._on_before_model_call
AfterModelCallEvent
    order=0   ModelRetryStrategy._handle_after_model_call
AfterInvocationEvent
    order=0   ModelRetryStrategy._handle_after_invocation
    order=0   plain function._on_after_invocation
BeforeToolsEvent
    order=0   AgentDelegation._on_before_tools
AfterToolCallEvent
    order=0   AgentDelegation._on_after_tool_call
AfterToolsEvent
    order=100   AgentDelegation._on_after_tools

9 callbacks were already registered, and none of them are ours.
SlidingWindowConversationManager is the conversation manager from Step 3:
it is a HookProvider too, and it compresses history from BeforeModelCallEvent.


### 5b. Watch the lifecycle fire, in order

This cell builds a `HookProvider` whose only job is to print each event as it arrives, then constructs an
agent with it and asks one question that requires a tool.

How it is put together:

- `block_types()` — a helper that reports what kind of content a message holds.
- `TracingHookProvider` — subscribes one recorder method to eight event types, so that everything arrives
  in a single numbered stream.
- Two `print` statements split the output into *construction* and *invocation*. That split is the
  experiment: whatever appears in the first section fired before anything was invoked.

`WATCHED` lists the events in the order I expected them. One of those expectations is wrong, and the
output will show which — the conclusion names it.

This cell makes two model calls, because a tool-using turn needs one call to request the tool and another
to interpret its result.

In [15]:
# =====================================================
# 5b. WATCHING THE LIFECYCLE FIRE
# =====================================================

# The event classes. Imported here rather than at the top of the notebook because
# this is the step where they start to mean something.
from strands.hooks import (
    AfterInvocationEvent,
    AfterModelCallEvent,
    AfterToolCallEvent,
    AgentInitializedEvent,
    BeforeInvocationEvent,
    BeforeModelCallEvent,
    BeforeToolCallEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)

# Listed in the order I expected them to fire. Treat it as a prediction to check
# against the output, not as documentation.
WATCHED = (
    AgentInitializedEvent,
    BeforeInvocationEvent,
    MessageAddedEvent,
    BeforeModelCallEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent,
    AfterModelCallEvent,
    AfterInvocationEvent,
)


def block_types(message):
    """List the kinds of content a message carries.

    A Strands message holds its content as a list of single-key dictionaries, such
    as {"text": "hello"} or {"toolUse": {...}}. That single key is the block's type,
    and watching it change across a conversation is what reveals a tool-using turn
    as four separate messages.

    Args:
        message: A message dict containing a "content" list.

    Returns:
        list[str]: One type name per content block, for example ["text"] or
            ["toolUse"] or ["toolResult"].
    """
    types = []

    for block in message["content"]:
        keys = list(block.keys())     # exactly one key per block
        types.append(keys[0])

    return types


# Subclassing HookProvider is optional: it is a Protocol, so any object with a
# register_hooks method would be accepted. Inheriting documents the intent and lets
# a type checker check the signature.
class TracingHookProvider(HookProvider):
    """Prints every lifecycle event it receives, in the order Strands fires them.

    One recorder method is subscribed to many event types, which puts all the
    events into a single numbered stream so the ordering is easy to read.
    """

    def __init__(self):
        """Create the provider with an empty event log.

        A provider is an object rather than a bare function so that it can hold
        state shared across callbacks. Here that state is the log, used to number
        the output lines and count the total. demo.py uses the same slot to hold
        its memory client and memory id.
        """
        self.events_seen = []

    def register_hooks(self, registry):
        """Subscribe this provider's recorder to every event type in WATCHED.

        Strands calls this method for you, once, while the Agent is being
        constructed: passing hooks=[provider] makes it call
        registry.add_hook(provider), which calls this.

        Args:
            registry: The agent's HookRegistry, supplied by Strands.

        Returns:
            None. The effect is that self.record becomes subscribed to each event.
        """
        for event_type in WATCHED:
            # The event type is passed explicitly. Passing None instead would make
            # add_callback infer the type from the callback's annotation, which
            # would fail here because record() is deliberately unannotated so that
            # it can accept any of the eight event classes.
            registry.add_callback(event_type, self.record)

    def record(self, event):
        """Print one numbered line describing an event that has just fired.

        Args:
            event: Any of the event objects listed in WATCHED. Left unannotated on
                purpose, since this one method handles all eight types.

        Returns:
            None. The effects are the printed line and the appended log entry.
        """
        name = type(event).__name__
        self.events_seen.append(name)

        position = len(self.events_seen)
        label = f"{position:>3}. {name}"      # :>3 keeps 1-9 aligned with 10+

        # Only MessageAddedEvent has a `message` field; the other seven inherit
        # nothing but `agent`, so reading .message on them would raise
        # AttributeError. Hence the check before reading it.
        if isinstance(event, MessageAddedEvent):
            role = event.message["role"]
            kinds = block_types(event.message)
            print(label, "  role=" + role, " blocks=" + str(kinds))
        else:
            print(label)


# The gap between these two print statements is the experiment. Anything printed
# before "now invoking it" fired during construction, not during the call.
print("--- constructing the agent (nothing invoked yet) ---")

tracer = TracingHookProvider()
traced = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[search_flights],
    hooks=[tracer],
)

print()
print("--- now invoking it ---")

# The return value is discarded because only the ordering matters here. The model's
# answer still appears, streamed to stdout by the default callback handler, which is
# why it shows up interleaved with the numbered lines rather than after them.
traced("Are there any flights from LHR to CDG on 2026-03-15?")

print()
print(len(tracer.events_seen), "events fired in total")

--- constructing the agent (nothing invoked yet) ---
  1. AgentInitializedEvent

--- now invoking it ---
  2. BeforeInvocationEvent
  3. MessageAddedEvent   role=user  blocks=['text']
  4. BeforeModelCallEvent


2026-09-25 16:07:44,672 [INFO] WanderBot.Notebook: search_flights(origin=LHR, destination=CDG, date=2026-03-15)



Tool #1: search_flights
  5. AfterModelCallEvent
  6. MessageAddedEvent   role=assistant  blocks=['toolUse']
  7. BeforeToolCallEvent
  8. AfterToolCallEvent
  9. MessageAddedEvent   role=user  blocks=['toolResult']
 10. BeforeModelCallEvent
Great news! There are two flights available from London Heathrow (LHR) to Paris Charles de Gaulle (CDG) on March 15, 2026:

**Flight Options:**
1. **HZ-101** - Departs 06:30, arrives 09:00 (local time)
   - Economy fare: $189.99
   - 42 seats available
   - Status: Scheduled

2. **HZ-102** - Departs 12:45, arrives 15:15 (local time)  
   - Economy fare: $159.99
   - Only 18 seats left!
   - Status: Scheduled

The second flight (HZ-102) offers the better price, but with limited seats remaining, you might want to book soon if this timing works for you. Would you like help with booking one of these flights or checking return options? 11. AfterModelCallEvent
 12. MessageAddedEvent   role=assistant  blocks=['text']
 13. AfterInvocationEvent

13 events 

### 5c. Rule one — the initialisation event fires at construction time

`AgentInitializedEvent` does not wait for you to invoke the agent. It fires on the last line of
`Agent.__init__`.

This cell proves it by checking a flag immediately before and immediately after `Agent(...)`, without ever
invoking the agent. That timing is what gives the memory read hook its opening: the agent is fully built,
the system prompt is assembled, and no model call has happened yet — so the prompt can still be changed.

Free: constructing an agent makes no API call.

In [16]:
# =====================================================
# 5c. RULE 1 - AgentInitializedEvent fires during construction
# =====================================================


class ConstructionProbe(HookProvider):
    """Records whether AgentInitializedEvent has fired, and does nothing else."""

    def __init__(self):
        """Create the probe with its flag unset."""
        self.fired = False

    def register_hooks(self, registry):
        """Subscribe to AgentInitializedEvent only.

        Args:
            registry: The agent's HookRegistry, supplied by Strands.

        Returns:
            None.
        """
        registry.add_callback(AgentInitializedEvent, self.on_init)

    def on_init(self, event):
        """Record that the event fired.

        Must be a plain def rather than async def - rule three below shows what
        happens otherwise.

        Args:
            event: The AgentInitializedEvent. Unused; only its arrival matters.

        Returns:
            None. Sets self.fired to True.
        """
        self.fired = True


probe = ConstructionProbe()
print("flag before Agent(...):", probe.fired)

# Construct the agent. Nothing is invoked: no question is asked, no model is called.
agent_for_probe = Agent(model=model, hooks=[probe])

print("flag after  Agent(...):", probe.fired)
print()
print("The event fired without anything being invoked, which is why the read hook")
print("can still edit system_prompt: no model call has happened yet.")

flag before Agent(...): False
flag after  Agent(...): True

The event fired without anything being invoked, which is why the read hook
can still edit system_prompt: no model call has happened yet.


### 5d. Rule two — the event is frozen, the agent behind it is not

Hook events reject writes by default. Each event class then opts *specific* fields back in, and the two
events used for memory opt in nothing at all — they are pure notifications.

This cell builds a `MessageAddedEvent` by hand, so we can try writing to it without running an agent, and
then makes two attempts:

1. Write to the event itself. This should raise `AttributeError`.
2. Read `event.agent` and write to the **agent**. This should succeed.

Both start with `event`, which is what makes the distinction easy to miss — and that second form is
exactly how the read hook injects prior turns into the prompt.

Free: no model call.

In [17]:
# =====================================================
# 5d. RULE 2 - events are read-only, the agent is not
# =====================================================

# Strands normally builds these. Constructing one by hand lets us try writing to it
# without running an agent at all.
sample_message = {"role": "user", "content": [{"text": "original text"}]}
event = MessageAddedEvent(agent=agent_for_probe, message=sample_message)

# ---- attempt 1: write to the event ----
replacement = {"role": "user", "content": [{"text": "tampered text"}]}

try:
    event.message = replacement
    print("attempt 1: writing event.message SUCCEEDED - unexpected")
except AttributeError as error:
    print("attempt 1: writing event.message ->", error)

# ---- attempt 2: read the event, then write to what it points at ----
print()
print("attempt 2: reading event.agent -> allowed, got a", type(event.agent).__name__)

event.agent.system_prompt = "Replaced by reaching through event.agent."

print("attempt 2: writing event.agent.system_prompt -> allowed")
print("           system_prompt is now:", event.agent.system_prompt)
print()
print("The event was never modified - it still points at the same agent.")

attempt 1: writing event.message -> Property message is not writable

attempt 2: reading event.agent -> allowed, got a Agent
attempt 2: writing event.agent.system_prompt -> allowed
           system_prompt is now: Replaced by reaching through event.agent.

The event was never modified - it still points at the same agent.


### 5e. Rule three — the initialisation callback must not be `async`

Because `AgentInitializedEvent` is dispatched from inside `Agent.__init__`, and a Python constructor cannot
`await`, Strands dispatches it synchronously. An `async def` callback would therefore never run: calling it
would just produce a coroutine object that nobody awaits.

Rather than let that fail silently, `add_callback` refuses the registration outright. This cell registers an
`async def` against that event and catches the `ValueError`.

Note the practical cost of this rule: reading prior turns out of AgentCore Memory is a network call, and it
has to happen inside a plain `def`, so it blocks while it runs.

Free: no model call, and no agent — a `HookRegistry` can be created on its own.

In [18]:
# =====================================================
# 5e. RULE 3 - AgentInitializedEvent refuses an async callback
# =====================================================


async def async_on_init(event):
    """An async callback, which AgentInitializedEvent will not accept.

    Args:
        event: The AgentInitializedEvent.

    Returns:
        None. The body never runs: registration fails before it can be called.
    """
    pass


# A registry can be built on its own, with no agent, which keeps this cheap.
standalone_registry = HookRegistry()

try:
    standalone_registry.add_callback(AgentInitializedEvent, async_on_init)
    print("the async callback was accepted - unexpected")
except ValueError as error:
    print("registering an async callback ->", error)

print()
print("A plain def is accepted by the same call:")


def sync_on_init(event):
    """A synchronous callback, which is accepted.

    Args:
        event: The AgentInitializedEvent.

    Returns:
        None.
    """
    pass


standalone_registry.add_callback(AgentInitializedEvent, sync_on_init)
print("  registered without complaint")

registering an async callback -> AgentInitializedEvent can only be registered with a synchronous callback

A plain def is accepted by the same call:
  registered without complaint


### What the output proved

One tool-using turn fired thirteen events, in this order:

```
  construction   1. AgentInitializedEvent
  invocation     2. BeforeInvocationEvent
                 3. MessageAddedEvent        role=user      blocks=['text']
                 4. BeforeModelCallEvent
                 5. AfterModelCallEvent
                 6. MessageAddedEvent        role=assistant blocks=['toolUse']
                 7. BeforeToolCallEvent
                 8. AfterToolCallEvent
                 9. MessageAddedEvent        role=user      blocks=['toolResult']
                10. BeforeModelCallEvent
                11. AfterModelCallEvent
                12. MessageAddedEvent        role=assistant blocks=['text']
                13. AfterInvocationEvent
```

**Event 1 arrived before the words "now invoking it" were printed.** `AgentInitializedEvent` is a
construction-time event, which is what gives the memory read hook its opportunity: the agent is fully
built, `system_prompt` is assembled, and no model call has happened yet.

**The prediction in `WATCHED` was wrong about the tools.** It placed `BeforeToolCallEvent` and
`AfterToolCallEvent` between the two model-call events, as though the tool ran *during* the model call. The
trace shows otherwise: `AfterModelCallEvent` at 5 comes before `BeforeToolCallEvent` at 7. The model call
**finishes**, returning a request to use a tool, and only then does the tool run — followed by a second
model call to interpret its result. A tool is never invoked inside a model call.

**`MessageAddedEvent` fired four times for one turn, not once.** These are exactly the four messages Step
2 found in `agent.messages`, now visible as four separate hook firings. Note the ordering: a message is
appended *after* the model call that produced it (5 then 6), and the tool result *after* the tool runs
(8 then 9).

That has a consequence for `demo.py` which is easy to miss. Its write hook pulls
`content[0].get("text")` and returns early when there is none — and events 6 and 9 carry a `toolUse` and
a `toolResult`, neither of which has a `text` key. So **only two of the four messages per turn are ever
persisted**: the user's question and the assistant's final answer. The tool call and its result are
dropped. Replayed history is therefore a clean transcript of the conversation, not a recording of how the
agent arrived at its answers — usually what you want, but a deliberate loss rather than an accident.

**The registry held nine callbacks before we added one.** Three framework components had already
subscribed: `SlidingWindowConversationManager` from Step 3, `ModelRetryStrategy`, and `AgentDelegation`.
One of them registers at `order=100` rather than the default `0`, so the ordering parameter is not
decoration — the framework uses it to run last. Hooks are not an extension point bolted onto Strands;
they are how Strands is assembled.

**And the three rules the SDK enforces, each with its real error text:**

| Rule | What happens |
| --- | --- |
| `AgentInitializedEvent` is construction-time | fired with no invocation made at all |
| Events are read-only | `AttributeError: Property message is not writable` |
| That event demands a sync callback | `ValueError: AgentInitializedEvent can only be registered with a synchronous callback` |

Yet `event.agent.system_prompt` accepted a write happily. The event is frozen; the agent it points at is
not. Reaching *through* an event is the whole basis of the read hook.

> **Note:** the trace watches eight event types, not all fifteen. `BeforeToolsEvent` and
> `AfterToolsEvent` fire around the tool *batch* and are visible in the registry cell but were left out
> of the trace to keep it readable. The model's streamed answer also appears interleaved with events 10
> and 11, because the default callback handler writes to stdout as tokens arrive.

Everything needed is now on the table. Step 3 established that history must live outside the agent
object; Step 4 put the two identifiers somewhere the model cannot see; Step 5 found the two moments where
code can run — once at construction to read, and on every message to write.

Step 6 assembles exactly that, against a Python dictionary rather than AWS. Proving the pattern before
introducing the service means that when AgentCore Memory does appear, the only new thing is the network
call.

## Step 6 — Memory that outlives the agent, without AWS

Everything needed is now established. Step 2 showed a new `Agent` starts with an empty history. Step 3
showed the manager trims what history there is. Step 4 put the two identifiers somewhere the model cannot
see. Step 5 found the two moments where code can run.

This step assembles them — but against a **Python dictionary** rather than AgentCore Memory. That is a
deliberate choice, not a shortcut. The pattern and the service are two separate things to learn, and
mixing them means that when something does not work you cannot tell which one is at fault. Prove the
pattern on a store you can read at a glance, and Part B's only new ingredient is the network call.

The dictionary is a fair stand-in because it implements the *same two methods*, with the same names,
arguments and return shapes as the real client:

| Method | Direction |
| --- | --- |
| `create_event(memory_id, actor_id, session_id, messages)` | write one or more messages |
| `get_last_k_turns(memory_id, actor_id, session_id, k)` | read the most recent turns back |

**One shape trap to know before reading the code.** Strands messages and AgentCore events do not look the
same, and the hook has to speak both:

``` Python
# a Strands message - role lower case, content is a LIST of blocks
{"role": "user", "content": [{"text": "Hello"}]}

# an AgentCore event - role UPPER case, content is a DICT
{"role": "USER", "content": {"text": "Hello"}}
```

That is why the write path reads `content[0]["text"]` while the read path reads `content["text"]`. It looks
like an inconsistency in `demo.py` until you notice the two sides are different formats.

### 6a. A memory store you can actually see inside

This cell defines `FakeMemoryStore`: a dictionary keyed by `(actor_id, session_id)`, holding a list of
events per conversation.

It implements only the two methods the hook needs, and deliberately builds records in the **real**
AgentCore shape — upper-case role, `content` as a dict. `get_last_k_turns` also groups messages into turns
the way the real client does: a new turn begins whenever a `USER` message arrives and something has
already been collected.

The cell ends with a **dry run** on a throwaway store, so you can see the exact inputs and outputs before
any agent is involved: what `create_event` accepts, what it stores, how `get_last_k_turns` groups records
into turns, what `k` cuts off, and what a different actor gets back.

Free — no model calls.

In [19]:
# =====================================================
# 6a. THE FAKE STORE
# =====================================================


class FakeMemoryStore:
    """A stand-in for AgentCore Memory that keeps everything in a dictionary.

    Implements the two MemoryClient methods the hook provider uses, with matching
    names, arguments and return shapes. Part B replaces this object with the real
    client, and the hook provider below does not have to change.
    """

    def __init__(self):
        """Create an empty store."""
        # Keys are (actor_id, session_id) tuples. Values are lists of event dicts,
        # oldest first. Two different actors therefore cannot see each other's
        # history, which is the isolation the real service gives you.
        self.events = {}

    def create_event(self, memory_id, actor_id, session_id, messages):
        """Append one or more messages to a conversation's history.

        Args:
            memory_id: Unused here. The real client uses it to choose the memory
                resource; this store only holds one.
            actor_id: Identifies the user the messages belong to.
            session_id: Identifies the conversation thread.
            messages: A list of (text, role) tuples, for example
                [("Hello", "USER")]. Role is case-insensitive.

        Returns:
            dict: A small acknowledgement. The real client returns the created
                event; nothing in this notebook reads the return value.
        """
        key = (actor_id, session_id)

        if key not in self.events:
            self.events[key] = []

        for text, role in messages:
            # Exactly the record the real client builds: role upper-cased, and
            # content a DICT with a "text" key - not a list of blocks.
            record = {"role": role.upper(), "content": {"text": text}}
            self.events[key].append(record)

        return {"stored": len(messages)}

    def get_last_k_turns(self, memory_id, actor_id, session_id, k=5):
        """Return the most recent turns of a conversation.

        A turn is one USER message plus whatever assistant messages follow it,
        which is how the real client groups them.

        Args:
            memory_id: Unused here.
            actor_id: Identifies the user.
            session_id: Identifies the conversation thread.
            k: Maximum number of turns to return.

        Returns:
            list[list[dict]]: A list of turns, each turn a list of message dicts
                shaped {"role": "USER", "content": {"text": "..."}}. An empty list
                when this actor and session have no history yet.
        """
        stored = self.events.get((actor_id, session_id), [])

        turns = []
        current_turn = []

        for record in stored:
            # A USER message starts a new turn - but only if the current turn has
            # something in it, otherwise the first message would open an empty one.
            if record["role"] == "USER" and current_turn:
                turns.append(current_turn)
                current_turn = []

            current_turn.append(record)

        if current_turn:
            turns.append(current_turn)

        # The most recent k turns, in chronological order.
        return turns[-k:]


# =====================================================
# 6a DRY RUN - exactly what goes in and what comes back
# =====================================================
# A throwaway store, so the one used in 6c still starts empty.
demo_store = FakeMemoryStore()

# ---- INPUT: create_event takes (text, role) TUPLES, not dicts ----
opening_turn = [
    ("Where can I fly to from London?", "USER"),
    ("Paris, Rome and Barcelona are all direct.", "ASSISTANT"),
]

print("create_event INPUT   messages =")
for pair in opening_turn:
    print("    ", pair)

written = demo_store.create_event(
    memory_id="demo-memory",
    actor_id="alice",
    session_id="s1",
    messages=opening_turn,
)
print("create_event OUTPUT ", written)

# ---- WHAT IT STORED: role upper-cased, content a DICT ----
print()
print("what ended up in the dictionary:")
print(json.dumps(demo_store.events[("alice", "s1")], indent=2))

print()
print("compare the two message shapes - the hook has to speak both:")
print("   stored here    :", {"role": "USER", "content": {"text": "..."}})
print("   Strands sends  :", {"role": "user", "content": [{"text": "..."}]})

# ---- add two more turns, the middle one with two assistant replies ----
demo_store.create_event(
    memory_id="demo-memory",
    actor_id="alice",
    session_id="s1",
    messages=[
        ("How much is Rome?", "USER"),
        ("About 145 dollars.", "ASSISTANT"),
        ("Shall I hold a seat?", "ASSISTANT"),
    ],
)
demo_store.create_event(
    memory_id="demo-memory",
    actor_id="alice",
    session_id="s1",
    messages=[
        ("Yes please.", "USER"),
        ("Held for 24 hours.", "ASSISTANT"),
    ],
)

total_records = len(demo_store.events[("alice", "s1")])
print()
print(total_records, "records are now stored. get_last_k_turns groups them into turns:")

# ---- OUTPUT: a list of turns, each turn a list of records ----
all_turns = demo_store.get_last_k_turns(
    memory_id="demo-memory",
    actor_id="alice",
    session_id="s1",
    k=5,
)

print("get_last_k_turns(k=5) returned", len(all_turns), "turns")
for turn_number, turn in enumerate(all_turns, start=1):
    print("    turn", turn_number, "-", len(turn), "message(s)")
    for record in turn:
        print("        ", record["role"], "->", record["content"]["text"])

print()
print("Turn 2 holds three messages: a turn is one USER message plus every")
print("assistant message that follows it, until the next USER message.")

# ---- k cuts from the recent end ----
recent = demo_store.get_last_k_turns(
    memory_id="demo-memory",
    actor_id="alice",
    session_id="s1",
    k=1,
)
print()
print("get_last_k_turns(k=1) returned", len(recent), "turn - the most recent:")
for record in recent[0]:
    print("        ", record["role"], "->", record["content"]["text"])

# ---- a different actor sees nothing, even with the same session id ----
bobs_view = demo_store.get_last_k_turns(
    memory_id="demo-memory",
    actor_id="bob",
    session_id="s1",
    k=5,
)
print()
print("same session_id but actor_id='bob' ->", bobs_view)
print("Empty, because the store is keyed on the PAIR (actor_id, session_id).")

create_event INPUT   messages =
     ('Where can I fly to from London?', 'USER')
     ('Paris, Rome and Barcelona are all direct.', 'ASSISTANT')
create_event OUTPUT  {'stored': 2}

what ended up in the dictionary:
[
  {
    "role": "USER",
    "content": {
      "text": "Where can I fly to from London?"
    }
  },
  {
    "role": "ASSISTANT",
    "content": {
      "text": "Paris, Rome and Barcelona are all direct."
    }
  }
]

compare the two message shapes - the hook has to speak both:
   stored here    : {'role': 'USER', 'content': {'text': '...'}}
   Strands sends  : {'role': 'user', 'content': [{'text': '...'}]}

7 records are now stored. get_last_k_turns groups them into turns:
get_last_k_turns(k=5) returned 3 turns
    turn 1 - 2 message(s)
         USER -> Where can I fly to from London?
         ASSISTANT -> Paris, Rome and Barcelona are all direct.
    turn 2 - 3 message(s)
         USER -> How much is Rome?
         ASSISTANT -> About 145 dollars.
         ASSISTANT -> Shal

### 6b. The hook provider — read once, write always

This is `demo.py`'s `ShortTermMemoryHookProvider`, with docstrings added and its logging replaced by plain
prints so the two paths are visible in the output.

Two callbacks, one per direction:

- **`on_agent_initialized`** — the read path. Fires once, during `Agent(...)`. Fetches recent turns and
  appends them to `system_prompt`. This is the whole illusion: history reaches the model as extra prompt
  text, nothing more.
- **`on_message_added`** — the write path. Fires on every message appended. Pulls the text out and stores
  it.

Both start by reading `actor_id` and `session_id` from `agent.state`, and both return immediately if either
is missing — so an agent without those ids still works, just without memory.

Watch for `[skip]` lines when this runs later. Step 5 showed a tool-using turn produces four messages, and
two of them carry a `toolUse` or `toolResult` block with no text at all.

Class definition only, so this cell is free.

In [20]:
# =====================================================
# 6b. THE HOOK PROVIDER
# =====================================================


class ShortTermMemoryHookProvider(HookProvider):
    """Gives an agent a memory that outlives the agent object.

    Subscribes to two lifecycle events, one per direction:
      - AgentInitializedEvent reads recent turns back and injects them
      - MessageAddedEvent writes each new message out
    """

    def __init__(self, store, memory_id, last_k_turns=5):
        """Hold the store and the settings shared by both callbacks.

        Args:
            store: Any object offering create_event and get_last_k_turns. Here a
                FakeMemoryStore; in Part B a real MemoryClient.
            memory_id: Identifier of the memory resource to read and write.
            last_k_turns: How many prior turns to replay. More turns means more
                context and more tokens on every invocation.
        """
        self.store = store
        self.memory_id = memory_id
        self.last_k_turns = last_k_turns

    def register_hooks(self, registry):
        """Subscribe both callbacks to their events.

        Args:
            registry: The agent's HookRegistry, supplied by Strands during
                construction.

        Returns:
            None.
        """
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized) # pairing with event is what define when method runs
        registry.add_callback(MessageAddedEvent, self.on_message_added)

    def on_agent_initialized(self, event):
        """Read recent turns and append them to the agent's system prompt.

        Runs once per invocation, during Agent(...), before any model call. That
        timing is what makes editing system_prompt safe here.

        Args:
            event: The AgentInitializedEvent. Used to reach event.agent, which is
                where both the ids and the prompt live.

        Returns:
            None. The effect is a longer system_prompt on the agent.
        """
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")

        # No ids means no memory. The agent still works; it just will not remember.
        if not actor_id or not session_id:
            return

        turns = self.store.get_last_k_turns(
            memory_id=self.memory_id,
            actor_id=actor_id,
            session_id=session_id,
            k=self.last_k_turns,
        )

        if not turns:
            print("  [read ] nothing stored yet for session", session_id)
            return

        transcript = []

        for turn in turns:
            for record in turn:
                role = record["role"].capitalize()
                # content is a DICT here, because this came from the store.
                text = record["content"]["text"]
                transcript.append(role + ": " + text)

        # The entire mechanism, in one line: prior history becomes prompt text.
        # Note this writes to event.agent, not to event - the event is read-only.
        event.agent.system_prompt = (
            event.agent.system_prompt + "\n\nRecent conversation:\n" + "\n".join(transcript)
        )

        print("  [read ] injected", len(transcript), "messages from", len(turns), "turn(s)")

    def on_message_added(self, event):
        """Write a newly added message to the store.

        Fires for every message the framework appends: the user's question, the
        assistant's reply, and also the tool-use and tool-result messages.

        Args:
            event: The MessageAddedEvent, carrying the message that was added.

        Returns:
            None. The effect is one stored record, or nothing when the message has
            no text to store.
        """
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")

        if not actor_id or not session_id:
            return

        role = event.message["role"]

        # content is a LIST of blocks here, because this is a Strands message.
        blocks = event.message["content"]
        first_block = blocks[0]

        # toolUse and toolResult blocks have no "text" key, so there is nothing to
        # store. This is why tool activity never reaches memory.
        if "text" not in first_block:
            print("  [skip ]", role, "message, blocks =", list(first_block.keys()))
            return

        self.store.create_event(
            memory_id=self.memory_id,
            actor_id=actor_id,
            session_id=session_id,
            messages=[(first_block["text"], role.upper())],
        )

        print("  [write]", role, "message stored")


print("ShortTermMemoryHookProvider defined.")

ShortTermMemoryHookProvider defined.


### 6c. Re-run Step 2's experiment, with memory attached

This is deliberately the *same* failing experiment from Step 2: ask about flights with one agent, then ask
"which one is the cheapest?" with a **different, freshly constructed** agent.

The only change is that both agents share a store and carry the same `session_id`. `build_remembering_agent`
mirrors `demo.py`'s entrypoint — a new agent per invocation, both ids in state, the memory hook attached.

What to expect. Invocation 1 prints `[read ] nothing stored yet`, then `[write]` lines as messages arrive.
Invocation 2 prints `[read ] injected ...` first, and should then answer **HZ-102 at $159.99** — the
question Step 2's second agent could not answer at all.

Watch for `[skip ]` lines during invocation 1, where the tool messages are passed over.

This cell makes model calls: two for the tool-using first question, one for the follow-up.

In [21]:
# =====================================================
# 6c. THE SAME EXPERIMENT, NOW WITH MEMORY
# =====================================================

store = FakeMemoryStore()
FAKE_MEMORY_ID = "fake-memory-001"
SESSION = "trip-planning-001"


def build_remembering_agent(actor_id, session_id):
    """Build a fresh agent wired to the shared store.

    Mirrors demo.py's entrypoint: a brand-new Agent per invocation, with both
    identifiers in state and the memory hook attached. The agent starts with an
    empty message list every time - memory comes from the store, not the object.

    Args:
        actor_id: Identifies the user.
        session_id: Identifies the conversation thread.

    Returns:
        Agent: A newly constructed agent. Constructing it fires the read hook.
    """
    return Agent(
        model=model,
        system_prompt=SYSTEM_PROMPT,
        tools=[search_flights],
        hooks=[ShortTermMemoryHookProvider(store, FAKE_MEMORY_ID)],
        state={"actor_id": actor_id, "session_id": session_id},
    )


print("=== invocation 1: a brand-new agent ===")
first_agent = build_remembering_agent("alice", SESSION)
first_agent(Q1)

print()
print("=== invocation 2: ANOTHER brand-new agent, same session ===")
second_agent = build_remembering_agent("alice", SESSION)
second_agent(Q2)

print()
print("second_agent started with", 0, "messages of its own;")
print("it now holds", len(second_agent.messages), "from this invocation only.")

=== invocation 1: a brand-new agent ===
  [read ] nothing stored yet for session trip-planning-001
  [write] user message stored


2026-09-25 16:07:47,139 [INFO] WanderBot.Notebook: search_flights(origin=LHR, destination=CDG, date=2026-03-15)



Tool #1: search_flights
  [skip ] assistant message, blocks = ['toolUse']
  [skip ] user message, blocks = ['toolResult']
Great news! There are two Horizon Travel flights from London Heathrow (LHR) to Paris Charles de Gaulle (CDG) on March 15, 2026:

**Flight Options:**

1. **HZ-101**  
   - Departure: 06:30  
   - Arrival: 09:00  
   - Class: Economy  
   - Price: $189.99  
   - Seats available: 42  
   - Status: SCHEDULED  

2. **HZ-102**  
   - Departure: 12:45  
   - Arrival: 15:15  
   - Class: Economy  
   - Price: $159.99  
   - Seats available: 18  
   - Status: SCHEDULED  

The afternoon flight (HZ-102) offers the best value at $159.99. Would you like help booking one of these flights or checking return options?  [write] assistant message stored

=== invocation 2: ANOTHER brand-new agent, same session ===
  [read ] injected 2 messages from 1 turn(s)
  [write] user message stored
The cheapest flight from London Heathrow (LHR) to Paris Charles de Gaulle (CDG) on March 15, 2026 

### 6d. Look in the store, and check the isolation

Two questions this cell answers by looking rather than by asserting.

**What actually got stored?** Printing every record shows which messages made it and which were skipped.
Compare the count against the number of messages the agents produced — Step 5 predicted that tool activity
would be dropped.

**Does `actor_id` really isolate?** A third agent uses the *same* `session_id` but a different actor. If
the key is genuinely `(actor_id, session_id)`, its read must come back empty, and it should be unable to
answer the follow-up — the leak your notes warn about, tested rather than assumed.

The store inspection is free. The isolation check makes one model call.

In [22]:
# =====================================================
# 6d. WHAT IS IN THE STORE, AND IS IT ISOLATED
# =====================================================


def show_store(store):
    """Print every record held in a FakeMemoryStore.

    Args:
        store: The FakeMemoryStore to inspect.

    Returns:
        None. Prints one line per stored record, grouped by conversation.
    """
    for key in store.events:
        actor_id, session_id = key
        records = store.events[key]

        print("actor =", actor_id, " session =", session_id, " records =", len(records))

        for position, record in enumerate(records):
            text = record["content"]["text"]
            preview = " ".join(text.split())[:60]
            print("   ", position, record["role"], "|", preview)


show_store(store)

print()
print("--- isolation: same session id, different actor ---")

other_agent = build_remembering_agent("bob", SESSION)
other_agent(Q2)

print()
print("conversations now in the store:", len(store.events))
print("keys:", list(store.events.keys()))

actor = alice  session = trip-planning-001  records = 4
    0 USER | Are there any flights from LHR to CDG on 2026-03-15?
    1 ASSISTANT | Great news! There are two Horizon Travel flights from London
    2 USER | Which one is the cheapest?
    3 ASSISTANT | The cheapest flight from London Heathrow (LHR) to Paris Char

--- isolation: same session id, different actor ---
  [read ] nothing stored yet for session trip-planning-001
  [write] user message stored
I can help you find the cheapest flight option, but I need a bit more information to proceed. Could you please tell me:

1. Your departure city (origin)
2. Your destination city
3. Your travel date

Once you provide these details, I'll search for available flights and show you the cheapest option for your trip!  [write] assistant message stored

conversations now in the store: 2
keys: [('alice', 'trip-planning-001'), ('bob', 'trip-planning-001')]


### What the output proved

**The pattern works, and it contains no AWS.** A freshly built agent answered a question that depended on
an exchange it never had, because the read hook turned stored history into prompt text before the first
model call. That is the whole mechanism — there is nothing else to it.

**Memory moved out of the object.** `second_agent` began with an empty message list, exactly as Step 2's
failing agent did. The difference is not that the agent kept anything; it is that something outside the
agent did, and handed it back at construction time.

**Tool activity is not stored, as Step 5 predicted.** The `[skip ]` lines show the `toolUse` and
`toolResult` messages being passed over, because neither carries a `text` block. So the store holds a clean
transcript of the conversation rather than a record of how the answers were produced. Usually desirable,
but a real loss: replay a session and you cannot see which tools ran.

**`actor_id` genuinely isolates.** Bob's agent used the same `session_id` as Alice's and still read nothing,
because the store is keyed on the pair. Get that wrong and every user shares one memory — which is exactly
the leak described in your notes, where a new conversation was told details had "already" been provided.

Two things to carry into Part B, both unverified here:

> **Note:** this store returns turns oldest-first. The real `get_last_k_turns` builds its list from
> `list_events` and returns `turns[:k]`, and I have not checked whether that list arrives newest-first. If
> it does, the injected transcript would be in reverse chronological order. Worth checking against the real
> service rather than assuming.

> **Note:** `create_event` here is instant. The real one is a network call inside a synchronous hook, so it
> blocks — and it fires on *every* message, meaning several round trips per turn.

What changes in Part B is one line: `FakeMemoryStore()` becomes `MemoryClient(region_name=REGION)`. The hook
provider, the entrypoint shape and the two ids all stay as they are. That is the point of having built it
this way round.

# Part B — the same pattern, against the real service

Step 6 built short-term memory against a Python dictionary on purpose. Moving to AgentCore Memory changes
exactly one line:

``` Python
store = FakeMemoryStore()                  # Part A
store = MemoryClient(region_name=REGION)   # Part B
```

The hook provider, the entrypoint shape and the two identifiers are all untouched. That is the dividend
from proving the pattern first: anything that misbehaves from here is the *service*, not the design.

**What a memory resource is.** AgentCore Memory is an AWS resource created independently of any agent. You
create it once, keep its id, and hand that id to whatever needs it — one resource serves many actors and
many sessions. Creation is asynchronous and takes a couple of minutes to reach `ACTIVE`.

**Short-term only.** `strategies=[]` means raw events are stored and nothing is extracted from them. That
choice also decides what you pay for: short-term events are one meter, while long-term storage and
retrieval are separate meters that only begin once a strategy exists.

**This part costs money, and leaves something behind.** Short-term memory is charged at $0.25 per 1,000
events. This notebook writes on the order of twenty, so under a cent — but the memory resource is real, and
Step 10 deletes it. Do not stop before that.

### 7a. Create the memory resource, or reuse it

This cell uses `create_or_get_memory` rather than `create_memory_and_wait`, because a notebook gets re-run.
Plain creation would produce a **new** resource on every run and orphan the previous ones; this reuses one
by name.

How it behaves, from the SDK source: it calls `create_memory_and_wait` first — so on a first run it blocks
until the resource is `ACTIVE`. If that fails with a `ValidationException` containing "already exists", it
falls back to `list_memories()` and picks the memory whose id starts with the name, returning immediately.

What to expect: an id of the form `<name>-<generated suffix>`, and status `ACTIVE`. The first run takes a
couple of minutes; later runs are instant.

> **Note:** the reuse path has two sharp edges. It matches on `id.startswith(name)`, so two memories whose
> names share a prefix could be confused with one another. And if no match is found it dereferences `None`,
> raising `TypeError` rather than saying anything useful about what went wrong.

The id is printed masked, because notebook outputs are saved into the `.ipynb` and committed with it.

In [23]:
# =====================================================
# 7a. CREATE (OR REUSE) THE MEMORY RESOURCE
# =====================================================
from bedrock_agentcore.memory import MemoryClient

MEMORY_NAME = "WanderBotNotebook"


def mask_memory_id(memory_id):
    """Hide the generated suffix of a memory id, keeping the readable name.

    A memory id is the resource name, a dash, then a generated suffix. The name is
    chosen by us and reveals nothing; the suffix identifies one specific resource,
    so that is the part withheld from output that gets committed.

    Args:
        memory_id: A memory id such as "WanderBotNotebook-ABC123XYZ".

    Returns:
        str: The id with its suffix replaced by asterisks, for example
            "WanderBotNotebook-*********". Falls back to mask_id when the id has
            no dash in it.
    """
    head, dash, tail = memory_id.rpartition("-")

    if dash:
        return head + dash + "*" * len(tail)
    return mask_id(memory_id)


memory_client = MemoryClient(region_name=REGION)

# strategies=[]        -> short-term only: raw events, no extraction pipeline, and
#                         none of the long-term meters.
# event_expiry_days=7  -> raw events are discarded after a week regardless.
memory = memory_client.create_or_get_memory(
    name=MEMORY_NAME,
    strategies=[],
    event_expiry_days=7,
)

# "id" is present whichever path ran. The create path and the reuse path return
# objects with different field sets, so read as little as possible from this one
# and ask for the status separately.
MEMORY_ID = memory["id"]

print("memory id :", mask_memory_id(MEMORY_ID))
print("status    :", memory_client.get_memory_status(MEMORY_ID))
print()
print("This is a live AWS resource. Step 10 deletes it - do not skip that cell.")

2026-09-25 21:05:17,421 [ERROR] bedrock_agentcore.memory.client: Failed to create memory: An error occurred (ValidationException) when calling the CreateMemory operation: Validation failed during CreateMemory: Memory with name WanderBotNotebook already exists


memory id : WanderBotNotebook-**********
status    : ACTIVE

This is a live AWS resource. Step 10 deletes it - do not skip that cell.


### 8a. Write a conversation, with no agent involved

Before wiring the service to an agent, it is worth driving it by hand. If a shape or an ordering is not
what you expected, you want to discover that with three lines of known input — not while also wondering
whether the hook fired, or whether the model behaved.

This cell writes three turns under a dedicated actor and session, kept separate from the agent demo in
Step 9 so the two cannot confuse each other. Each message is **numbered**, which is the whole trick: when
they come back, their order is unmistakable.

Note that each `create_event` call carries two messages. That is one event, not two — billing counts events,
so this cell writes three of them.

Costs three short-term events, well under a cent.

In [24]:
# =====================================================
# 8a. WRITE A KNOWN CONVERSATION, NO AGENT INVOLVED
# =====================================================

# A dedicated actor and session, so this probe cannot interfere with Step 9.
PROBE_ACTOR = "direct-probe"
PROBE_SESSION = "probe-001"

# Three turns. The numbers make the order unmistakable on the way back out.
probe_turns = [
    [("1. I want to go to Rome", "USER"), ("1. Lovely - when are you travelling?", "ASSISTANT")],
    [("2. How much is it?", "USER"), ("2. About 145 dollars.", "ASSISTANT")],
    [("3. Book it please", "USER"), ("3. Booked, reference HZ-77.", "ASSISTANT")],
]

for turn in probe_turns:
    # One call carrying two messages = ONE event with a two-item payload.
    memory_client.create_event(
        memory_id=MEMORY_ID,
        actor_id=PROBE_ACTOR,
        session_id=PROBE_SESSION,
        messages=turn,
    )
    print("wrote event containing:", turn[0][0], "/", turn[1][0])

print()
print(len(probe_turns), "events written, each holding 2 messages.")

wrote event containing: 1. I want to go to Rome / 1. Lovely - when are you travelling?
wrote event containing: 2. How much is it? / 2. About 145 dollars.
wrote event containing: 3. Book it please / 3. Booked, reference HZ-77.

3 events written, each holding 2 messages.


### 8b. Read it back, and settle the ordering question

Two read paths, and one open question from Step 6.

`list_events` returns the raw events, payloads and all. Printing one in full shows what the service actually
stores, as opposed to what the SDK hands back after processing.

`get_last_k_turns` is the one the hook uses. It groups messages into turns and returns a list of lists.

**The open question.** My `FakeMemoryStore` returned turns oldest-first. The real implementation builds its
list from `list_events` and returns `turns[:k]`, and I never checked which end `list_events` starts from. If
it is newest-first, then the transcript the read hook injects is in **reverse chronological order** — which
would still work, in the sense that the model gets the facts, but would be actively misleading about what
was said when.

The numbered messages answer it directly. The cell ends by checking whether turn 1 comes back first and
saying so in plain words, rather than leaving it for you to squint at.

Read-only, so no new events and no extra cost.

In [25]:
# =====================================================
# 8b. READ IT BACK, AND CHECK THE ORDER
# =====================================================

# ---- the raw view ----
raw_events = memory_client.list_events(
    memory_id=MEMORY_ID,
    actor_id=PROBE_ACTOR,
    session_id=PROBE_SESSION,
    max_results=10,
)

print("list_events returned", len(raw_events), "events")


def redacted(event):
    """Copy an event with its memory id masked, safe to leave in saved output.

    Notebook outputs are stored inside the .ipynb and committed with it, so a raw
    dump of a service response would publish the real resource id. Only memoryId
    needs hiding here: actorId and sessionId are values this notebook invented,
    and eventId identifies nothing outside the resource.

    Args:
        event: One event dict as returned by list_events.

    Returns:
        dict: A shallow copy with memoryId replaced by its masked form.
    """
    safe = dict(event)
    safe["memoryId"] = mask_memory_id(event["memoryId"])
    return safe


print()
print("the first event it returned, in full (memory id masked):")
# default=str because the event carries a datetime, which json cannot serialise.
print(json.dumps(redacted(raw_events[0]), indent=2, default=str))

# ---- the view the hook uses ----
turns = memory_client.get_last_k_turns(
    memory_id=MEMORY_ID,
    actor_id=PROBE_ACTOR,
    session_id=PROBE_SESSION,
    k=5,
)

print()
print("get_last_k_turns returned", len(turns), "turns, in this order:")

for position, turn in enumerate(turns, start=1):
    print("  position", position, "in the returned list:")
    for record in turn:
        print("      ", record["role"], "->", record["content"]["text"])


def describe_ordering(returned_turns):
    """Say whether the returned turns are in chronological order.

    Relies on the probe messages being numbered, so the first message of the
    first returned turn reveals which end of the conversation came back first.

    Args:
        returned_turns: The list of turns from get_last_k_turns.

    Returns:
        str: A sentence describing the observed ordering.
    """
    if not returned_turns:
        return "nothing came back, so there is nothing to conclude"

    first_message = returned_turns[0][0]["content"]["text"]
    turn_number = first_message[0]

    if turn_number == "1":
        return "CHRONOLOGICAL - turn 1 came back first, oldest to newest"
    return "REVERSED - turn " + turn_number + " came back first, so newest to oldest"


print()
print("ordering:", describe_ordering(turns))

list_events returned 6 events

the first event it returned, in full (memory id masked):
{
  "memoryId": "WanderBotNotebook-**********",
  "actorId": "direct-probe",
  "sessionId": "probe-001",
  "eventId": "0000001790360080364#839d64cc",
  "eventTimestamp": "2026-09-25 21:14:40.364000+03:00",
  "payload": [
    {
      "conversational": {
        "content": {
          "text": "3. Book it please"
        },
        "role": "USER"
      }
    },
    {
      "conversational": {
        "content": {
          "text": "3. Booked, reference HZ-77."
        },
        "role": "ASSISTANT"
      }
    }
  ],
  "branch": {
    "name": "main"
  }
}

get_last_k_turns returned 5 turns, in this order:
  position 1 in the returned list:
       USER -> 3. Book it please
       ASSISTANT -> 3. Booked, reference HZ-77.
  position 2 in the returned list:
       USER -> 2. How much is it?
       ASSISTANT -> 2. About 145 dollars.
  position 3 in the returned list:
       USER -> 1. I want to go to Rome
 

### What the output proved

**The stored event is richer than the SDK's convenience methods suggest.** A raw event carries `memoryId`,
`actorId`, `sessionId`, `eventId`, `eventTimestamp`, the `payload`, and a `branch` — defaulting to
`{"name": "main"}`, which is a hint that AgentCore Memory supports conversation branching. The `eventId`
looks like `0000001790359829448#82f3e902`: a zero-padded, sortable prefix followed by a hash, which is
presumably how the service orders events without needing to compare timestamps.

**And `get_last_k_turns` returns turns newest-first.** This is the question Step 6 left open, and the answer
is the less convenient one:

```
position 1  ->  turn 3   "Book it please"
position 2  ->  turn 2   "How much is it?"
position 3  ->  turn 1   "I want to go to Rome"
```

`list_events` returns most-recent-first, and `get_last_k_turns` builds its list in that order and returns
`turns[:k]`. The name promises the last *k* turns and delivers exactly that — it just hands them back
backwards.

**Which makes `demo.py`'s read hook subtly wrong.** It iterates the returned turns in order and joins them
into a transcript, so the text injected into the system prompt reads:

```
Recent conversation:
User: 3. Book it please
Assistant: 3. Booked, reference HZ-77.
User: 2. How much is it?
...
User: 1. I want to go to Rome
```

The model still receives every fact, which is why nothing looks broken — and with only one prior turn there
is no order to get wrong at all, which is why Step 6 passed and why this survives a two-turn demo. It starts
to matter as soon as the conversation is long enough for sequence to carry meaning: "book the second one",
"actually, change that", "the cheaper flight". A model reading those in reverse is being actively misled
about what was said when.

The fix is one word in the loop:

``` Python
for turn in reversed(recent_turns):     # oldest first, as a transcript should read
```

Step 9 uses the corrected version rather than copying the defect, and says so where it does.

> **Note:** worth noticing *how* this was caught. Numbering the messages before writing them cost nothing
> and made the ordering impossible to misread. Had this step used realistic text, the reversed transcript
> would have looked plausible and been missed — which is the argument for probing a service by hand before
> wiring it to anything.